In [21]:
#create only the main list
import json
import re
from pathlib import Path
from typing import List
import pandas as pd

# =========================
# CONFIGURE THESE PATHS
# =========================
SNAPSHOT_DIR = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots")
OUTPUT_DIR   = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\Method_V2.0")
MAIN_COMMIT_LIST = OUTPUT_DIR / "Main_Commit_List.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# NORMALIZATION / HELPERS
# =========================
URL_RE = re.compile(r"http\S+|www\.\S+", re.I)

def normalize_subject(s: str) -> str:
    """Lowercase, strip URLs/emojis/control chars, collapse whitespace."""
    s = (s or "").lower()
    s = URL_RE.sub("", s)
    s = re.sub(r"[\u0000-\u001f\u007f]", " ", s)                 # control chars
    s = re.sub(r"[\U00010000-\U0010FFFF]", "", s)                # emojis / astral symbols
    s = re.sub(r"\s+", " ", s).strip()                           # collapse spaces
    return s

def load_snapshots(folder: Path) -> pd.DataFrame:
    """
    Read all .jsonl files and dedupe to one row per (repo, sha).
    Keep 'subject_raw' and a normalized 'subject_norm'.
    """
    rows = []
    for p in folder.glob("*.jsonl"):
        with p.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    d = json.loads(line)
                except Exception:
                    continue
                repo = d.get("repo") or p.stem
                sha  = d.get("sha")
                if not sha:
                    continue
                subj = d.get("subject") or d.get("commit_raw") or d.get("message") or ""
                rows.append({"repo": repo, "sha": sha, "subject_raw": subj})

    if not rows:
        return pd.DataFrame(columns=["repo", "sha", "subject_raw", "subject_norm"])

    df = pd.DataFrame(rows)
    # Prefer rows with non-empty subject if duplicates
    df["has_subj"] = df["subject_raw"].fillna("").ne("")
    df = (
        df.sort_values(["repo", "sha", "has_subj"], ascending=[True, True, False])
          .drop_duplicates(subset=["repo", "sha"], keep="first")
          .drop(columns=["has_subj"])
    )
    df["subject_norm"] = df["subject_raw"].apply(normalize_subject)
    return df

# =========================
# MAIN
# =========================
def main():
    df = load_snapshots(SNAPSHOT_DIR)
    print(f"[INFO] Commits loaded (deduped): {len(df)}")

    # Minimal commit list only
    cols = ["repo", "sha", "subject_raw", "subject_norm"]
    df[cols].to_csv(MAIN_COMMIT_LIST, index=False, encoding="utf-8")
    print(f"[OK] Wrote main commit list: {MAIN_COMMIT_LIST} (rows={len(df)})")

if __name__ == "__main__":
    main()



[INFO] Commits loaded (deduped): 106597
[OK] Wrote main commit list: C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\Method_V2.0\Main_Commit_List.csv (rows=106597)


## update the main list

In [28]:
import re
from pathlib import Path
from typing import List, Dict
import pandas as pd

# =========================
# CONFIGURE THESE PATHS
# =========================
BASE_DIR = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\Method_V2.0")
MAIN_COMMIT_LIST = BASE_DIR / "Main_Commit_List.csv"                # input
DETECTION_LIST   = BASE_DIR / "detection_list.csv"                  # input
OUTPUT_FILE      = BASE_DIR / "Main_Commmit_List_Updated.csv"       # output (3 m's)

SEP = "|"  # joiner for multi-values

def split_keywords_colon(s: str) -> List[str]:
    """Split detection_list keywords by colon only, trim, drop empties."""
    if not isinstance(s, str):
        return []
    # allow spaces around colons, tolerate accidental quotes
    parts = [p.strip().strip('"').strip("'") for p in s.split(":")]
    return [p for p in parts if p]

def choose_text_column(df: pd.DataFrame) -> str:
    """Choose the commit text column to search."""
    for col in ["subject_norm", "subject_raw", "subject", "message"]:
        if col in df.columns:
            return col
    df["subject_norm"] = ""  # fallback
    return "subject_norm"

def main():
    # Read inputs (utf-8-sig tolerates BOM from Excel)
    df = pd.read_csv(MAIN_COMMIT_LIST, encoding="utf-8-sig")
    det = pd.read_csv(DETECTION_LIST,   encoding="utf-8-sig")

    # Normalize detector headers & filter usable rows
    det.columns = [c.strip().lower() for c in det.columns]
    if "intent" not in det.columns or "keywords" not in det.columns:
        raise ValueError("detection_list.csv must have 'intent' and 'keywords' columns.")

    if "enabled" in det.columns:
        det["enabled"] = (
            det["enabled"].fillna(True).astype(str).str.strip().str.lower()
            .isin(["1","true","t","yes","y"])
        )
        det = det[det["enabled"]]

    det = det[(det["intent"].astype(str).str.strip() != "") &
              (det["keywords"].astype(str).str.strip() != "")].copy()

    # Expand to (intent, keyword) rows using COLON splitting
    rows: List[Dict[str, str]] = []
    for _, r in det.iterrows():
        intent = str(r["intent"]).strip()
        for kw in split_keywords_colon(r["keywords"]):
            rows.append({"intent": intent, "keyword": kw})
    if not rows:
        # no detectors; write blank columns and exit
        df["intent"] = ""
        df["keywords"] = ""
        df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")
        print(f"[WARN] No usable detectors. Wrote {OUTPUT_FILE} with empty columns.")
        return

    kw_df = pd.DataFrame(rows).drop_duplicates()
    kw_df["kw_lc"] = kw_df["keyword"].astype(str).str.lower()

    # Choose text column and lower-case it for matching
    text_col = choose_text_column(df)
    text_lc = df[text_col].fillna("").astype(str).str.lower()

    intents_out, keywords_out = [], []

    # Brute-force substring matching (case-insensitive)
    kws = list(zip(kw_df["intent"].to_list(), kw_df["keyword"].to_list(), kw_df["kw_lc"].to_list()))
    for txt in text_lc.tolist():
        matched_intents = set()
        matched_keywords = set()
        for intent, kw_orig, kw_lc in kws:
            if kw_lc and kw_lc in txt:
                matched_intents.add(intent)
                matched_keywords.add(kw_orig)  # keep original casing
        intents_out.append(SEP.join(sorted(matched_intents)) if matched_intents else "")
        keywords_out.append(SEP.join(sorted(matched_keywords)) if matched_keywords else "")

    # Attach and save
    df["intent"] = intents_out
    df["keywords"] = keywords_out
    df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")
    print(f"[OK] Wrote {OUTPUT_FILE} with columns: intent, keywords")

if __name__ == "__main__":
    main()


[OK] Wrote C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\Method_V2.0\Main_Commmit_List_Updated.csv with columns: intent, keywords


In [29]:
import pandas as pd
import random
from pathlib import Path

# =========================
# CONFIG
# =========================
UPDATED_CSV = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\Method_V2.0\Main_Commmit_List_Updated.csv")
OUTPUT_DIR  = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\Method_V2.0")

ITERATION          = 3       # bump each round (e.g., 3, 4, 5, ...)
SAMPLE_SIZE        = 300     # how many to export
SEED               = 12345   # fixed seed for reproducibility
STRATIFY_BY_REPO   = False   # True = proportional per repo

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# HELPERS
# =========================
def sample_rows(df_in: pd.DataFrame, n: int, seed: int, stratify: bool) -> pd.DataFrame:
    """Sample n rows from df_in (optionally stratified by repo)."""
    if df_in.empty:
        return df_in
    n = min(n, len(df_in))
    if not stratify:
        return df_in.sample(n=n, random_state=seed)

    # Proportional by repo
    rng = random.Random(seed)
    parts = []
    total = len(df_in)
    for repo, g in df_in.groupby("repo", sort=False):
        k = max(1, round(n * (len(g) / total)))
        parts.append(g.sample(n=min(k, len(g)), random_state=rng.randint(0, 10**9)))
    out = pd.concat(parts, ignore_index=True)
    if len(out) > n:
        out = out.sample(n=n, random_state=seed)  # trim to exact n
    return out

def choose_subject_columns(df: pd.DataFrame):
    """Return the subject columns present (keep order preference)."""
    cols = []
    if "subject_raw" in df.columns:  cols.append("subject_raw")
    if "subject_norm" in df.columns: cols.append("subject_norm")
    return cols

# =========================
# MAIN
# =========================
def main():
    # 1) Load updated main list
    df = pd.read_csv(UPDATED_CSV, encoding="utf-8-sig")

    required_cols = {"repo", "sha", "intent"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Main_Commmit_List_Updated is missing required columns: {sorted(missing)}")

    # 2) Filter rows where intent is BLANK (empty, whitespace, or NaN)
    intent_blank_mask = df["intent"].fillna("").astype(str).str.strip().eq("")
    df_blank_intent = df.loc[intent_blank_mask].copy()

    print(f"[INFO] Total rows: {len(df):,}")
    print(f"[INFO] Blank-intent rows: {len(df_blank_intent):,}")

    # 3) Sample from blank-intent rows
    sample = sample_rows(df_blank_intent, SAMPLE_SIZE, SEED, STRATIFY_BY_REPO)

    # 4) Save sample
    out_path = OUTPUT_DIR / f"Sample_To_Review_iter{ITERATION}.csv"
    subject_cols = choose_subject_columns(df)
    base_cols = ["repo", "sha"]
    sample_cols = base_cols + subject_cols  # keep lean for manual review
    # ensure columns exist (they should, but be safe)
    sample_cols = [c for c in sample_cols if c in sample.columns]

    sample[sample_cols].to_csv(out_path, index=False, encoding="utf-8")
    print(f"[OK] Sample written: {out_path}  (rows={len(sample)})")

    # 5) Sanity check: all sampled rows should have blank intent
    leaked = (~sample["intent"].fillna("").astype(str).str.strip().eq("")).sum() if not sample.empty else 0
    print(f"[OK] Verified sampling from blank-intent only (leaked={leaked}).")

if __name__ == "__main__":
    main()


[INFO] Total rows: 106,597
[INFO] Blank-intent rows: 58,444
[OK] Sample written: C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\Method_V2.0\Sample_To_Review_iter3.csv  (rows=300)
[OK] Verified sampling from blank-intent only (leaked=0).
